In [ ]:
# =========================================================
# CONFIGURAÇÃO LOCAL
# REMOVER ANTES DE SUBIR PARA O GITHUB
# =========================================================

import os

JAVA_HOME = (
    r"C:\Users\GCarapinadelima\Downloads"
    r"\microsoft-jdk-17.0.20.1-windows-x64"
    r"\jdk-17.0.20.1+1"
)

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent

OUTPUT_DIR = SCRIPT_DIR.parent / "outputs" / "gold_06"
GRAFICOS_DIR = SCRIPT_DIR.parent / "graficos" / "gold_06"

GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# ARQUIVOS
# ---------------------------------------------------------------------

ARQUIVO_SENIORIDADE = OUTPUT_DIR / "06_02_salario_por_senioridade.csv"
ARQUIVO_MODELO = OUTPUT_DIR / "06_04_senioridade_x_modelo_trabalho.csv"
ARQUIVO_REGIAO = OUTPUT_DIR / "06_05_senioridade_x_regiao.csv"


# ---------------------------------------------------------------------
# VALIDAR ARQUIVOS
# ---------------------------------------------------------------------

arquivos_necessarios = [
    ARQUIVO_SENIORIDADE,
    ARQUIVO_MODELO,
    ARQUIVO_REGIAO
]

for arquivo in arquivos_necessarios:
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {arquivo}"
        )


# ---------------------------------------------------------------------
# CARREGAR OUTPUTS
# ---------------------------------------------------------------------

df_senioridade = pd.read_csv(
    ARQUIVO_SENIORIDADE,
    sep=";",
    encoding="utf-8-sig"
)

df_modelo = pd.read_csv(
    ARQUIVO_MODELO,
    sep=";",
    encoding="utf-8-sig"
)

df_regiao = pd.read_csv(
    ARQUIVO_REGIAO,
    sep=";",
    encoding="utf-8-sig"
)


# ---------------------------------------------------------------------
# ORDENAÇÕES
# ---------------------------------------------------------------------

ORDEM_NIVEL = [
    "Júnior",
    "Pleno",
    "Sênior",
    "Especialista/Staff+"
]

ORDEM_MODELO = [
    "100% presencial",
    "Híbrido com dias fixos",
    "Híbrido flexível",
    "100% remoto"
]

ORDEM_REGIAO = [
    "Norte",
    "Nordeste",
    "Centro-Oeste",
    "Sudeste",
    "Sul"
]


# ---------------------------------------------------------------------
# LABELS DAS FAIXAS SALARIAIS
# ---------------------------------------------------------------------

LABELS_FAIXAS = {
    1: "< R$ 1 mil",
    2: "R$ 1–2 mil",
    3: "R$ 2–3 mil",
    4: "R$ 3–4 mil",
    5: "R$ 4–6 mil",
    6: "R$ 6–8 mil",
    7: "R$ 8–12 mil",
    8: "R$ 12–16 mil",
    9: "R$ 16–20 mil",
    10: "R$ 20–25 mil",
    11: "R$ 25–30 mil",
    12: "R$ 30–40 mil",
    13: "> R$ 40 mil"
}


# ---------------------------------------------------------------------
# CONFIGURAÇÃO GERAL
# ---------------------------------------------------------------------

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 18,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11
    }
)


# ---------------------------------------------------------------------
# FUNÇÃO PARA SALVAR GRÁFICO
# ---------------------------------------------------------------------

def salvar_grafico(nome_arquivo):
    caminho = GRAFICOS_DIR / nome_arquivo

    plt.savefig(
        caminho,
        dpi=300,
        bbox_inches="tight"
    )

    print(f"Gráfico salvo: {caminho}")

    plt.show()
    plt.close()


# ---------------------------------------------------------------------
# FAIXA SALARIAL MEDIANA POR SENIORIDADE
# ---------------------------------------------------------------------

df_grafico_1 = df_senioridade.copy()

df_grafico_1["nivel"] = pd.Categorical(
    df_grafico_1["nivel"],
    categories=ORDEM_NIVEL,
    ordered=True
)

df_grafico_1 = (
    df_grafico_1
    .sort_values(["edicao", "nivel"])
)

fig, ax = plt.subplots(figsize=(12, 7))

for edicao in sorted(df_grafico_1["edicao"].unique()):
    base_edicao = (
        df_grafico_1[
            df_grafico_1["edicao"] == edicao
        ]
        .sort_values("nivel")
    )

    ax.plot(
        base_edicao["nivel"].astype(str),
        base_edicao["ordem_faixa_mediana"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        label=edicao
    )

    for _, linha in base_edicao.iterrows():
        ax.annotate(
            LABELS_FAIXAS.get(
                int(linha["ordem_faixa_mediana"]),
                ""
            ),
            (
                str(linha["nivel"]),
                linha["ordem_faixa_mediana"]
            ),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=10
        )

ax.set_title(
    "Faixa salarial mediana por senioridade",
    loc="left",
    pad=28,
    fontweight="bold"
)

ax.text(
    0,
    1.02,
    "Comparação entre as edições 2024–2025 e 2025–2026",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Senioridade")
ax.set_ylabel("Faixa salarial mediana")

ax.set_yticks(
    list(LABELS_FAIXAS.keys())
)

ax.set_yticklabels(
    list(LABELS_FAIXAS.values())
)

ax.grid(
    axis="y",
    alpha=0.25
)

ax.legend(
    title="Edição",
    frameon=False
)

plt.tight_layout()

salvar_grafico(
    "06_01_evolucao_salario_senioridade.png"
)


# ---------------------------------------------------------------------
# SENIORIDADE X MODELO DE TRABALHO
# ---------------------------------------------------------------------

EDICAO_ATUAL = "2025-2026"

df_grafico_2 = (
    df_modelo[
        (
            df_modelo["edicao"]
            == EDICAO_ATUAL
        )
        &
        (
            df_modelo["status_amostra"]
            == "OK"
        )
    ]
    .copy()
)

df_grafico_2["nivel"] = pd.Categorical(
    df_grafico_2["nivel"],
    categories=ORDEM_NIVEL,
    ordered=True
)

df_grafico_2["modelo_trabalho"] = pd.Categorical(
    df_grafico_2["modelo_trabalho"],
    categories=ORDEM_MODELO,
    ordered=True
)

matriz_modelo = (
    df_grafico_2
    .pivot_table(
        index="modelo_trabalho",
        columns="nivel",
        values="ordem_faixa_mediana",
        observed=False
    )
    .reindex(
        index=ORDEM_MODELO,
        columns=ORDEM_NIVEL
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

imagem = ax.imshow(
    matriz_modelo.values,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(ORDEM_NIVEL))
)

ax.set_xticklabels(
    ORDEM_NIVEL
)

ax.set_yticks(
    np.arange(len(ORDEM_MODELO))
)

ax.set_yticklabels(
    ORDEM_MODELO
)

for i in range(matriz_modelo.shape[0]):
    for j in range(matriz_modelo.shape[1]):
        valor = matriz_modelo.iloc[i, j]

        if pd.notna(valor):
            label = LABELS_FAIXAS.get(
                int(valor),
                ""
            )

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold"
            )

        else:
            ax.text(
                j,
                i,
                "n < 30",
                ha="center",
                va="center",
                fontsize=10
            )

ax.set_title(
    "Faixa salarial por senioridade e modelo de trabalho",
    loc="left",
    pad=28,
    fontweight="bold"
)

ax.text(
    0,
    1.02,
    "Faixa salarial mediana | 2025–2026 | grupos com n ≥ 30",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Senioridade")
ax.set_ylabel("Modelo de trabalho")

plt.colorbar(
    imagem,
    ax=ax,
    label="Ordem da faixa salarial"
)

plt.tight_layout()

salvar_grafico(
    "06_02_salario_senioridade_modelo_trabalho.png"
)


# ---------------------------------------------------------------------
# SENIORIDADE X REGIÃO
# ---------------------------------------------------------------------

df_grafico_3 = (
    df_regiao[
        (
            df_regiao["edicao"]
            == EDICAO_ATUAL
        )
        &
        (
            df_regiao["status_amostra"]
            == "OK"
        )
    ]
    .copy()
)

df_grafico_3["nivel"] = pd.Categorical(
    df_grafico_3["nivel"],
    categories=ORDEM_NIVEL,
    ordered=True
)

df_grafico_3["regiao_onde_mora"] = pd.Categorical(
    df_grafico_3["regiao_onde_mora"],
    categories=ORDEM_REGIAO,
    ordered=True
)

matriz_regiao = (
    df_grafico_3
    .pivot_table(
        index="regiao_onde_mora",
        columns="nivel",
        values="ordem_faixa_mediana",
        observed=False
    )
    .reindex(
        index=ORDEM_REGIAO,
        columns=ORDEM_NIVEL
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

imagem = ax.imshow(
    matriz_regiao.values,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(ORDEM_NIVEL))
)

ax.set_xticklabels(
    ORDEM_NIVEL
)

ax.set_yticks(
    np.arange(len(ORDEM_REGIAO))
)

ax.set_yticklabels(
    ORDEM_REGIAO
)

for i in range(matriz_regiao.shape[0]):
    for j in range(matriz_regiao.shape[1]):
        valor = matriz_regiao.iloc[i, j]

        if pd.notna(valor):
            label = LABELS_FAIXAS.get(
                int(valor),
                ""
            )

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold"
            )

        else:
            ax.text(
                j,
                i,
                "n < 30",
                ha="center",
                va="center",
                fontsize=10
            )

ax.set_title(
    "Faixa salarial por senioridade e região",
    loc="left",
    pad=28,
    fontweight="bold"
)

ax.text(
    0,
    1.02,
    "Faixa salarial mediana | 2025–2026 | grupos com n ≥ 30",
    transform=ax.transAxes,
    fontsize=12
)

ax.set_xlabel("Senioridade")
ax.set_ylabel("Região")

plt.colorbar(
    imagem,
    ax=ax,
    label="Ordem da faixa salarial"
)

plt.tight_layout()

salvar_grafico(
    "06_03_salario_senioridade_regiao.png"
)


# ---------------------------------------------------------------------
# FINALIZAÇÃO
# ---------------------------------------------------------------------

print("\n" + "=" * 120)
print("GRÁFICOS DA GOLD 06 FINALIZADOS")
print("=" * 120)

print(
    f"\nPasta de saída: {GRAFICOS_DIR}"
)